# Experiment 2 — one additional POTER seed, existing baselines reused

**Cost-limited plan: POTER seeds1/2 total, ERM_enhanced seed0 and DPE_legacy seed0.**
Seed1 is already complete. This notebook trains only seed2, from ImageNet V1,
using the original 300-epoch, batch128, FP32, OT-weighted sampling recipe.
No extra probe training, third POTER seed, ERM or DPE fitting is scheduled.

**Writing/opening this notebook does not authorize training.** Training is skipped by
default. Notify/agree with the user on cloud compute before changing the confirmation flag.
Local 6 GB GPU is insufficient for the historical peak10.6 GiB. Use a cloud GPU with
at least16 GiB. Historical comparable-T4 estimate: about9.1 GPU-hours plus setup/evaluation;
no new training benchmark or measured speedup is claimed.

Historical Waterbirds training source bytes are embedded here; newline-only encoding
differences are restored to audited hashes. One unused CelebA module differs and is
recorded explicitly; its historical bytes were not recovered. No CUB algorithm is edited.
No Git clone or current branch download is required. Preserve the original framework versions
below for primary replication. If unavailable, discuss the deviation before training.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, zipfile, io, base64, subprocess, shutil
from datetime import datetime, timezone

CONFIRM_TRAINING = False  # Set True only after user confirmation of this cloud job.
SEED = 2                 # Exactly one additional seed; never initialize from seed1 weights.
PROFILE_PHASES = False   # Optional host/phase timings during the authorized full run.
MAX_HOURS = 9.5          # Session budget; original runner pauses only at epoch boundaries.
ROOT = Path('/kaggle/working/experiment2-poter-source')
DATA = Path('/kaggle/working/poter-data')
FEATURE = Path('/kaggle/working/poter-features/waterbirds_standard.pt')
RESULTS = Path('/kaggle/working/poter-results')
FEATURE_INPUT = None     # Optional attached exact waterbirds_standard.pt path.
DATA_INPUT = None        # Optional extracted waterbird_complete95_forest2water2 directory.
RESTORE_ARCHIVE = None   # Optional trusted full prior output ZIP, before resuming seed2.

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4*1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def preserve_write(path, payload):
    path = Path(path)
    if path.exists():
        if path.read_bytes() != payload:
            raise ValueError(f'Existing file differs; preserved, not overwritten: {path}')
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(payload)


In [ ]:
PAYLOAD = '...'
REFERENCE = json.loads('{"dataset": "CUB", "seed": 1, "epochs": 300, "batch_size": 128, "lr": 1e-05, "weight_decay": 1.0, "f_power": 0.1, "clip_cap": 20.0, "reg_scale": 0.03, "augment_data": true, "torch": "2.11.0+cu130", "torchvision": "0.26.0+cu130", "numpy": "2.1.3", "metadata_sha256": "2f023b9dc371f6fbb82c7cb810ff0f03e19c54c8055a2e37517ae7bb56644362", "cache_sha256": "8a7636923bffaddce16f843b38afe0a97743f19a151339ddb96e200d2c4d2258", "source_sha256": {"train.py": "776603ba16ff00cd27c9075cb2ef6703b41c062eda33c4443e6e4fe4b105431a", "loss.py": "5640227087f82ad7234b8dc39a2b823a7590e4f24a39c3706bdfbc5fe80872d2", "ot_reweight.py": "6b2925c33a12f01404e1a27bf471fb08f8177fb0e3e2068de47588e7785a6f0b", "utils.py": "2299323bbe0c0e1e9c0afea725024d026922c37b3bb97caaa9dda577f8fcf0ba", "run_expt.py": "f0b34fde7a0d26f5bf37604b2b139923d2b8de17e1a6d9dff91dd546054aa01b", "scripts/kaggle_waterbirds.py": "b6e1af63a6a4f6635643c112e4a7c3a66e505775ac2acf2d512ac13dcd5e1dc2", "scripts/training_checkpoint.py": "7a275546a6351e31a84a8b377f1280909964fd940125f19e4bd23e57815e2ecb", "data/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "data/celebA_dataset.py": "9691652c35e6baf1b3a43db1cb2eb1f3992a84f599d3ca2c8006ab1482101e31", "data/civilcomments_dataset.py": "263c1bd7382aed1f35b7974a2ff4474b3b1df7b40974ce362a5ab9165073a9c2", "data/cmnist_dataset.py": "c18bc17447c49f89aab942693f91b9ca64a00beca3afb120a41588b7fc5e5a19", "data/confounder_dataset.py": "3d0e865b9750bd8e417f80df8d6cdd7bc26bc12177d695b3f443d07a0fb2b584", "data/confounder_utils.py": "02d2799602b172fc73a6c55a75f6e79e136517c5c28a53a1f3171dcddc2360d2", "data/cub_dataset.py": "464f3a69e485100d614182d87bef19c9aad1eb4cf3fde8cbe61ea4958e2fea87", "data/data.py": "18b4d7c8eeeda15f6c2afec97ed98c22fba1d0c003c52193c5d33d1648683ee2", "data/dro_dataset.py": "90a9c56fb6a7a887ee51d5a55eebf773662034ad2026520444c4ada68f195bce", "data/label_shift_utils.py": "68273e0ebda071bca1f66cd91b182a411a9ae7083acb23e761d6603b0acf9caf"}}')
SOURCE_HASHES = json.loads('{"data/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "data/celebA_dataset.py": "86a9b05e60b9e0e77b58083279bc170754c569698f56b7b50d832a044ed25470", "data/civilcomments_dataset.py": "263c1bd7382aed1f35b7974a2ff4474b3b1df7b40974ce362a5ab9165073a9c2", "data/cmnist_dataset.py": "c18bc17447c49f89aab942693f91b9ca64a00beca3afb120a41588b7fc5e5a19", "data/confounder_dataset.py": "3d0e865b9750bd8e417f80df8d6cdd7bc26bc12177d695b3f443d07a0fb2b584", "data/confounder_utils.py": "02d2799602b172fc73a6c55a75f6e79e136517c5c28a53a1f3171dcddc2360d2", "data/cub_dataset.py": "464f3a69e485100d614182d87bef19c9aad1eb4cf3fde8cbe61ea4958e2fea87", "data/data.py": "18b4d7c8eeeda15f6c2afec97ed98c22fba1d0c003c52193c5d33d1648683ee2", "data/dro_dataset.py": "90a9c56fb6a7a887ee51d5a55eebf773662034ad2026520444c4ada68f195bce", "data/label_shift_utils.py": "68273e0ebda071bca1f66cd91b182a411a9ae7083acb23e761d6603b0acf9caf", "extract_features.py": "acfe6e202924cc980b6b2ff8eb99de588435acaae85ce6bb0a5b2fa455d46620", "loss.py": "5640227087f82ad7234b8dc39a2b823a7590e4f24a39c3706bdfbc5fe80872d2", "models.py": "04213ebb1a934a2d06809464fef22861b2463e7419ac40604fa5b238e33400b9", "noise.py": "f55ad0d3f1d1fd3bfa46fe180aec89dbfc3d046669c75f282743e9ae88834d47", "ot_reweight.py": "6b2925c33a12f01404e1a27bf471fb08f8177fb0e3e2068de47588e7785a6f0b", "run_expt.py": "f0b34fde7a0d26f5bf37604b2b139923d2b8de17e1a6d9dff91dd546054aa01b", "scripts/kaggle_waterbirds.py": "b6e1af63a6a4f6635643c112e4a7c3a66e505775ac2acf2d512ac13dcd5e1dc2", "scripts/summarize_waterbirds.py": "96cf26f1d1e9b757866893c291a811c31c199cd09eb383ac3e2392dea5a51f51", "scripts/training_checkpoint.py": "7a275546a6351e31a84a8b377f1280909964fd940125f19e4bd23e57815e2ecb", "train.py": "776603ba16ff00cd27c9075cb2ef6703b41c062eda33c4443e6e4fe4b105431a", "utils.py": "2299323bbe0c0e1e9c0afea725024d026922c37b3bb97caaa9dda577f8fcf0ba", "utils_glue.py": "4751a5cfe2745a2d6c07444089f734c38841af574a6b1c1214bc388ace240d1a", "profile_poter_training.py": "ed6c83d7bffd05825a2db704cb3fd0907a1eebe8959c2c42af3be50d678ea6e2"}')
SOURCE_DEVIATIONS = json.loads('{"data/celebA_dataset.py": {"historical_sha256": "9691652c35e6baf1b3a43db1cb2eb1f3992a84f599d3ca2c8006ab1482101e31", "included_sha256": "86a9b05e60b9e0e77b58083279bc170754c569698f56b7b50d832a044ed25470", "reason": "Unrelated CelebA dataset module; not the Waterbirds/CUB implementation. Historical bytes not recovered."}}')
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    for name in archive.namelist():
        target = (ROOT / name).resolve()
        assert target.is_relative_to(ROOT.resolve()), name
        preserve_write(target, archive.read(name))
for name, expected in SOURCE_HASHES.items():
    assert digest(ROOT / name) == expected, f'Source fingerprint differs: {name}'
for name, expected in REFERENCE['source_sha256'].items():
    assert name in SOURCE_DEVIATIONS or digest(ROOT/name) == expected
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch, torchvision, numpy as np, pandas as pd
environment = dict(torch=str(torch.__version__), torchvision=str(torchvision.__version__), numpy=np.__version__)
env_matches = all(environment[key] == REFERENCE[key] for key in environment)
print('Current:', environment)
print('Required:', {key:REFERENCE[key] for key in environment})
print('Environment matches:', env_matches, '; training enabled:', CONFIRM_TRAINING)
print('Historical source deviations:', SOURCE_DEVIATIONS)
print('Waterbirds/CUB training sources match historical fingerprints; unused CelebA deviation recorded.')


## Restore progress first

Only restore your own trusted output ZIP. It must include the whole run directory,
`last_training.pt`, `best_validation.pt`, config and metric CSVs. A selected checkpoint
alone cannot resume optimizer/RNG state. Existing different files are preserved and cause
an error. Restore into a fresh working directory rather than overwriting live progress.
A seed1 snapshot cannot resume seed2. Runtime/source/config must match the saved run.


In [ ]:
import stat
if RESTORE_ARCHIVE is not None:
    with zipfile.ZipFile(RESTORE_ARCHIVE) as archive:
        for item in archive.infolist():
            if item.is_dir(): continue
            target = (RESULTS / item.filename).resolve()
            assert target.is_relative_to(RESULTS.resolve()), item.filename
            assert not stat.S_ISLNK(item.external_attr >> 16), 'ZIP symlinks not accepted'
            preserve_write(target, archive.read(item))
    print('Trusted archive restored without overwriting different files.')
run_dir = RESULTS / 'CUB' / 'Clean' / f'POTER_CUB_seed{SEED}_kaggle'
state_path = run_dir / 'last_training.pt'
if state_path.exists():
    state = torch.load(state_path, map_location='cpu', weights_only=False)
    assert all(k in state for k in ['model','optimizer','rng','loss_computer','epoch','config'])
    assert state['config']['seed'] == SEED and state['config']['epochs'] == 300
    assert state['config']['batch_size'] == 128
    print('Resume from completed epoch', state['epoch'], '; same seed only.')
    print('Approx remaining hours:', max(0, 299-state['epoch'])*108.65/3600)
    del state
elif run_dir.exists() and any(run_dir.iterdir()):
    raise RuntimeError('Existing run without resume state: preserve/recover it before training.')
else:
    print('No saved seed2 state. Fresh seed2 will use original ImageNet initialization.')


## Reuse exact dataset and fixed OT cache

Attach the existing `waterbirds_standard.pt` cache (about61 MB) as a private Kaggle input.
The currently audited local `D:/POTER_features/waterbirds_standard.pt` has a different hash
and must **not** be assumed compatible. Recover the exact cache from the original cloud run.
The notebook discovers that name under `/kaggle/input` and accepts it only if its SHA matches.
It never
recomputes OT features as an untracked fallback. Train/validation/test counts must be
4795/1199/5794. The official dataset archive is downloaded only if no extracted dataset
is available. Nothing here uses test accuracy to choose settings or checkpoints.


In [ ]:
import urllib.request, tarfile
target = DATA / 'cub/data/waterbird_complete95_forest2water2'
if DATA_INPUT is not None and not target.exists():
    source = Path(DATA_INPUT)
    assert digest(source/'metadata.csv') == REFERENCE['metadata_sha256']
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, target)  # Stage unchanged files on local cloud disk.
if not (target/'metadata.csv').exists():
    archive = DATA.parent/'waterbirds.tar.gz'
    if not archive.exists():
        temporary = archive.with_suffix('.download')
        urllib.request.urlretrieve('https://downloads.cs.stanford.edu/nlp/data/dro/waterbird_complete95_forest2water2.tar.gz', temporary)
        assert digest(temporary) == '56c51b77f7d17283ac97003d973e9107e8af9ceebffcb0c8be14d024f846f951'
        temporary.replace(archive)
    assert digest(archive) == '56c51b77f7d17283ac97003d973e9107e8af9ceebffcb0c8be14d024f846f951'
    destination = target.parent
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as tar:
        for item in tar:
            path = (destination/item.name).resolve()
            assert path.is_relative_to(destination.resolve()) and (item.isfile() or item.isdir())
            if item.isdir(): path.mkdir(parents=True, exist_ok=True)
            else: preserve_write(path, tar.extractfile(item).read())
assert digest(target/'metadata.csv') == REFERENCE['metadata_sha256']
metadata = pd.read_csv(target/'metadata.csv')
assert metadata['split'].value_counts().to_dict() == {2:5794, 0:4795, 1:1199}
assert set(metadata['y']) == {0,1} and set(metadata['place']) == {0,1}
assert all((target/name).is_file() for name in metadata['img_filename'])
if not FEATURE.exists():
    candidates = [Path(FEATURE_INPUT)] if FEATURE_INPUT is not None else list(Path('/kaggle/input').rglob('waterbirds_standard.pt'))
    matching = [p for p in candidates if p.is_file() and digest(p) == REFERENCE['cache_sha256']]
    if not matching:
        raise FileNotFoundError('Attach the exact original waterbirds_standard.pt cache, then rerun this cell.')
    FEATURE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(matching[0], FEATURE)
assert digest(FEATURE) == REFERENCE['cache_sha256']
print('Dataset and fixed OT cache verified. Existing predictions require no rerun.')


## Optional timing and the single training job

`PROFILE_PHASES=True` wraps unchanged `run_epoch` calls and times DataLoader host wait
plus train/validation/test wall time. It does not change batches, precision, loss or
optimizer; synchronization adds timing overhead. This is not a CUDA kernel profile.
AMP and extra workers are **not enabled**: they have no validated parity/speedup here.
Selection stays strictly best validation WGA. The original runner logs test metrics
each epoch; those metrics are not selection inputs. No epoch reduction or early stopping.

For a runtime version mismatch, recreate the original environment before confirming:
torch2.11.0+cu130, torchvision0.26.0+cu130, NumPy2.1.3. No automatic package upgrade is done.


In [ ]:
if not CONFIRM_TRAINING:
    print('SKIPPED: training not confirmed. No optimizer/model training is launched.')
else:
    assert SEED == 2, 'Only seed2 is prioritized; third seed is deferred.'
    assert env_matches, 'Primary replication requires the original environment versions.'
    assert torch.cuda.is_available(), 'Cloud CUDA GPU required.'
    assert torch.cuda.get_device_properties(0).total_memory >= 12*1024**3, 'Need at least12 GiB visible VRAM; use a16 GB class GPU or larger.'
    assert digest(FEATURE) == REFERENCE['cache_sha256']
    marker = RESULTS.parent/'experiment2-seed2-active-process.json'
    if marker.exists():
        previous = json.loads(marker.read_text())
        assert Path('/proc').exists(), 'Cannot verify prior process; inspect runtime before launching another job.'
        commandline = Path(f"/proc/{previous['pid']}/cmdline")
        if commandline.exists():
            running = commandline.read_bytes().replace(b'\0',b' ').decode(errors='replace')
            assert str(run_dir) not in running, 'Prior seed2 process is still running; do not launch a duplicate.'
    arguments = ['--data-dir',str(DATA),'--cache',str(FEATURE),'--run-dir',str(run_dir),
                 '--seed','2','--epochs','300','--batch-size','128','--max-hours',str(MAX_HOURS)]
    command = [sys.executable,str(ROOT/'scripts/kaggle_waterbirds.py'),*arguments]
    if PROFILE_PHASES:
        command = [sys.executable,str(ROOT/'profile_poter_training.py'),'--confirm-training',
                   '--baseline-root',str(ROOT),'--profile-output',str(run_dir/'phase_profile.json'),'--',*arguments]
    print('Launching one seed; full original recipe:', command)
    process = subprocess.Popen(command, cwd=ROOT)
    marker.write_text(json.dumps(dict(pid=process.pid,run_dir=str(run_dir),command=command),indent=2))
    try:
        returncode = process.wait()
        if returncode: raise subprocess.CalledProcessError(returncode,command)
    except KeyboardInterrupt:
        print('Wait interrupted. Child PID', process.pid, 'was NOT terminated. Inspect it before rerunning/closing the session.')
        raise
    finally:
        if process.poll() is None:
            print('Job still running: no termination and no live-state ZIP snapshot attempted. Wait for an epoch checkpoint and job exit before backup.')
        elif RESULTS.exists():
            stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
            backup = shutil.make_archive(str(RESULTS.parent/f'experiment2-seed2-backup-{stamp}'), 'zip', RESULTS)
            print('SAVE/DOWNLOAD this full archive before session ends:', backup)
            print('Archive SHA256:', digest(backup))
    print('Run returned. It may have paused at an epoch boundary; verify completion below.')


## Verify completion and export progress

Rerunning resumes a matching partial seed2. A session-budget pause is **not** completion.
Keep `last_training.pt`, best checkpoint, config and logs together. `/kaggle/working`
alone is not a durable backup: save the notebook output version or download the ZIP.
A hard runtime kill can lose uncommitted work; never stop a known live job before
verifying an accessible durable full-state checkpoint. The notebook does not kill jobs.


In [ ]:
if state_path.exists():
    state = torch.load(state_path, map_location='cpu', weights_only=False)
    assert state['config']['seed'] == 2 and state['config']['epochs'] == 300
    completed = state['epoch'] == 299
    record = dict(seed=2,last_completed_epoch=int(state['epoch']),training_complete=completed,
                  multi_seed_validation_complete=False,selection='strictly best validation WGA',
                  precision='FP32',epochs=300,batch_size=128,
                  last_training_sha256=digest(state_path),
                  best_validation_sha256=digest(run_dir/'best_validation.pt'),
                  next_action='Import selected checkpoint and run locked 2100-image evaluation' if completed else 'Restore full state and resume same seed')
    del state
    (run_dir/'export_status.json').write_text(json.dumps(record,indent=2))
    print(json.dumps(record,indent=2))
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    backup = shutil.make_archive(str(RESULTS.parent/f'experiment2-seed2-final-{stamp}'), 'zip', RESULTS)
    print('Download complete state bundle:', backup, '\nSHA256:', digest(backup))
else:
    print('No seed2 progress: training was skipped or did not commit an epoch. Seed1 alone remains available.')


## Evaluation handoff — still pending

Return the full seed2 bundle to the repository. Evaluate its validation-selected checkpoint
on the exact locked 2,100 inputs (1,600 natural +400 gray +100 original; sigma0/.5/1/2).
Reuse all6,300 existing POTER seed1/ERM seed0/DPE seed0 predictions. No new baseline inference
is needed. This notebook does **not** produce those counterfactual predictions or claim evaluation.

Report both POTER seeds and two-seed mean±sample SD separately from foreground-bootstrap CIs.
Two seeds give preliminary stability evidence only; the original three-seed target remains deferred.
ERM/DPE seed variability is not established by one baseline checkpoint. Do not silently train
a third seed based on test outcomes or tune the method on the repeatedly inspected test cohort.
